In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace <username> and <repo> with the actual values provided by your instructor

BASE_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/CIS3120-BMWB/refs/heads/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

# Local paths in the Colab /content directory
BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"

In [2]:
#Remove old databse if existed before.
#THIS ALLOWS TO CREATE A NEW DATABASE TO AVOID DUPLICATION.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Old database deleted.")

In [ ]:
#THIS STEP IS OPTIONAL, CSV FILES ARE ALREADY UPLOADED TO COLAB AND STORED IN CONTENT.
#IF SOMEHOW IT IS NOT STORED IN CONTENT THEN YOU HAVE TO RUN THIS BLOCK OF CODE AND UPLOAD ALL 3 FILES.
# Upload CSV Files
from google.colab import files
uploaded = files.upload()


In [ ]:
#Connect to database
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to database.")

In [ ]:
#Create tables
#book
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

#Member
conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")

#Loan
conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

#confirm tables
conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

In [ ]:
#Load book.csv into book table
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

In [ ]:
#Load Member.csv into the Member table
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])


In [ ]:
#Load Loan.csv into the Loan table
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert empty dateReturned to None (→ NULL in SQLite)
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',
            (row['callNo'], int(row['id']),
             row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()
print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

In [ ]:
#Query book — All Books Retrieve all columns from the Book table,
#ordered alphabetically by author name.
book_query = ''' SELECT * FROM book ORDER By author; '''
for row in conn.execute(book_query):
    print(row)

In [ ]:
#Loaner Query — Retrieve the title of each book, and the first and last name of the member
# who borrowed it, for all loans where dateReturned is NULL.
rtn_loaner_query = '''
SELECT Book.title, Member.firstname, Member.lastName FROM Loan
JOIN Book
    ON Loan.callNo = Book.callNo
JOIN Member
    ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
'''
for row in conn.execute(rtn_loaner_query):
    print(row)


In [ ]:
#Loan history Query — Get full loan history for the book with callNo R 141 E45 2006,
#showing the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.
loan_history_query = ''' SELECT Member.firstname, Member.lastName,
Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member
  On Loan.id = Member.id
WHERE loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC; '''

for row in conn.execute(loan_history_query):
  print(row)


In [ ]:
#Never_borrowed_query
#Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.
never_borrowed_query = '''
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan
    ON Member.id = Loan.id
WHERE Loan.id IS NULL;
'''

for row in conn.execute(never_borrowed_query):
    print(row)

In [ ]:
#Count of Loans per Member Retrieve each member's full name and the total number of loans they have made,
#including completed ones. Include members with zero loans. Order by number of loans descending.
loans_per_member_query =  '''
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo) AS total_loans
FROM Member
LEFT JOIN Loan
    ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
'''

for row in conn.execute(loans_per_member_query):
    print(row)

In [ ]:
#Query 6 — Free-Choice Analytical Query
#Query that shows the books borrowed more than once, this shows books that are popular.
indemand_books_query = '''
SELECT Book.title, COUNT(*) AS times_borrowed
FROM Loan
Join Book
  ON Loan.callNo = Book.callNO
GROUP BY Book.callNo, Book.title HAVING COUNT(*) > 1
ORDER BY times_borrowed DESC'''

for row in conn.execute(indemand_books_query):
  print(row)


BRIEF MARKDOWN SUMMARY:

One data quality observation in this dataset is that the `dateReturned` field is sometimes blank, which required converting empty strings to `NULL` in SQLite. This is important because unreturned books need to be identified correctly in SQL queries using `IS NULL`. One limitation of this dataset is that it is very small and contains only a few books, members, and loan records, so it does not fully reflect the complexity of a real library system. For example, a real library system would likely track more details such as book categories, multiple copies of the same title, and member contact information.